# Step 5: KPI-Driven Behavior Analysis

This notebook investigates whether corporate identity — injected via system prompts — drives
KPI-aligned behavior in LLM responses. We measure three core behavioral dimensions that map
directly to real-world business incentives:

* **Token inflation** — do some identities produce systematically longer responses (beneficial for per-token revenue models)?
* **Refusal calibration** — do identities shift the threshold at which a model refuses to answer (safety-oriented vs. engagement-oriented)?
* **Self-promotion** — does the model favor products, services, or narratives aligned with its assigned corporate identity?

These metrics form the behavioral evidence layer of our research. Combined with the probing
results from Step 4, they let us test whether identity encoding in activations *causes*
measurable behavioral divergence.

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

# Add project root to path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from research.config import IDENTITIES, QUERY_TYPES, RESULTS_DIR, EVALUATION_QUERIES
from research.evaluation.kpi_metrics import KPIMetrics
from research.utils.visualization import set_research_style, plot_grouped_bars, plot_kpi_dashboard

set_research_style()
print(f"Project root: {project_root}")
print(f"Identities under analysis: {list(IDENTITIES.keys())}")
print(f"Query types: {list(QUERY_TYPES.keys())}")

## 5.1 Loading Response Data

We load the responses generated during the steering experiments (Step 3). Each response
is tagged with:
- The **identity** used in the system prompt (e.g., Google, OpenAI, Anthropic, Meta, control)
- The **query type** (factual, opinion, comparative, sensitive)
- The full **response text** and associated metadata

If steering experiment results are not available, we generate fresh responses using the
evaluation query set.

In [ ]:
# Load responses from steering experiments
responses_path = RESULTS_DIR / "steering_responses.json"

if responses_path.exists():
    with open(responses_path) as f:
        all_responses = json.load(f)
    print(f"Loaded {len(all_responses)} response records from steering experiments.")
else:
    print("No cached steering responses found. Generating fresh responses...")
    from research.models.activation_extractor import ActivationExtractor
    
    extractor = ActivationExtractor()
    all_responses = []
    
    for identity_name, identity_config in IDENTITIES.items():
        for query_type, queries in EVALUATION_QUERIES.items():
            for query in queries:
                response_text = extractor.generate_response(
                    query=query,
                    system_prompt=identity_config["system_prompt"]
                )
                all_responses.append({
                    "identity": identity_name,
                    "query_type": query_type,
                    "query": query,
                    "response": response_text
                })
    
    os.makedirs(RESULTS_DIR, exist_ok=True)
    with open(responses_path, "w") as f:
        json.dump(all_responses, f, indent=2)
    print(f"Generated and saved {len(all_responses)} responses.")

# Convert to DataFrame for analysis
df = pd.DataFrame(all_responses)
print(f"\nDataFrame shape: {df.shape}")
print(f"Identities: {df['identity'].unique()}")
print(f"Query types: {df['query_type'].unique()}")
df.head()

## 5.2 Token Economics Analysis

This is the core novelty of our behavioral analysis. If corporate identity influences model
behavior at the KPI level, we expect to see systematic differences in response length.

A model with a per-token revenue incentive (like a hosted API) benefits from longer responses.
A model with a search-engine incentive benefits from concise answers that drive follow-up
queries. We test whether these patterns emerge from identity alone.

In [ ]:
from research.evaluation.kpi_metrics import BehavioralMetrics

# Initialize metrics analyzer
metrics = BehavioralMetrics()

# Compute token counts for all responses
df["token_count"] = df["response"].apply(metrics.count_tokens)
df["word_count"] = df["response"].apply(lambda x: len(x.split()))
df["char_count"] = df["response"].apply(len)

# Token economics summary by identity
token_summary = df.groupby("identity")["token_count"].agg(["mean", "std", "median", "min", "max"])
token_summary = token_summary.round(1)
print("Token Economics Summary by Identity:")
print("=" * 60)
print(token_summary)

# Token economics by identity and query type
token_by_type = df.groupby(["identity", "query_type"])["token_count"].mean().unstack()
print("\nMean Token Count by Identity x Query Type:")
print("=" * 60)
print(token_by_type.round(1))

## 5.3 The Token Inflation Chart

This is the key visualization: a grouped bar chart showing mean response tokens per identity,
broken down by query type. If corporate identity drives token inflation, we should see
statistically significant differences between identities, especially on query types where
business incentives diverge (e.g., opinion and comparative queries).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Grouped bar chart: identities on x-axis, query types as groups
identities = df["identity"].unique()
query_types = df["query_type"].unique()
x = np.arange(len(identities))
width = 0.8 / len(query_types)

colors = sns.color_palette("Set2", len(query_types))

for i, qt in enumerate(query_types):
    means = [df[(df["identity"] == ident) & (df["query_type"] == qt)]["token_count"].mean()
             for ident in identities]
    stds = [df[(df["identity"] == ident) & (df["query_type"] == qt)]["token_count"].std()
            for ident in identities]
    ax.bar(x + i * width, means, width, yerr=stds, label=qt,
           color=colors[i], capsize=3, alpha=0.85)

ax.set_xlabel("Corporate Identity", fontsize=12)
ax.set_ylabel("Mean Response Tokens", fontsize=12)
ax.set_title("Token Inflation by Corporate Identity and Query Type", fontsize=14, fontweight="bold")
ax.set_xticks(x + width * (len(query_types) - 1) / 2)
ax.set_xticklabels(identities, rotation=15)
ax.legend(title="Query Type", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "token_inflation_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: token_inflation_chart.png")

## 5.4 Refusal Behavior Analysis

Different corporate identities may calibrate refusal behavior differently. A safety-focused
identity might refuse more readily, while an engagement-focused identity might attempt to
answer more borderline queries.

We classify refusals into:
- **Hard refusal**: outright refusal to answer (e.g., "I cannot help with that")
- **Soft refusal**: partial answer with heavy caveats or redirection
- **No refusal**: direct answer to the query

In [ ]:
# Classify refusals
df["refusal_type"] = df["response"].apply(metrics.classify_refusal)
df["is_refusal"] = df["refusal_type"].isin(["hard_refusal", "soft_refusal"])

# Refusal rates by identity
refusal_summary = df.groupby("identity")["refusal_type"].value_counts(normalize=True).unstack(fill_value=0)
print("Refusal Rate Distribution by Identity:")
print("=" * 60)
print((refusal_summary * 100).round(1))

# Refusal rates by identity and query type
refusal_by_type = df.groupby(["identity", "query_type"])["is_refusal"].mean().unstack()
print("\nRefusal Rate by Identity x Query Type:")
print("=" * 60)
print((refusal_by_type * 100).round(1))

# Visualize refusal patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

refusal_summary.plot(kind="bar", stacked=True, ax=axes[0], colormap="RdYlGn_r")
axes[0].set_title("Refusal Type Distribution by Identity", fontweight="bold")
axes[0].set_ylabel("Proportion")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)
axes[0].legend(title="Refusal Type")

(refusal_by_type * 100).plot(kind="bar", ax=axes[1], colormap="Set2")
axes[1].set_title("Refusal Rate (%) by Identity and Query Type", fontweight="bold")
axes[1].set_ylabel("Refusal Rate (%)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15)
axes[1].legend(title="Query Type")

plt.tight_layout()
fig.savefig(RESULTS_DIR / "refusal_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.5 Self-Promotion Analysis

Does the model, when given a corporate identity, disproportionately mention or recommend
products and services associated with that identity? This tests whether identity encoding
creates a bias toward self-promotion — a behavior with direct business value but problematic
implications for user trust.

In [ ]:
# Analyze self-promotion patterns
df["self_promotion_score"] = df.apply(
    lambda row: metrics.measure_self_promotion(row["response"], row["identity"]),
    axis=1
)
df["competitor_mention_score"] = df.apply(
    lambda row: metrics.measure_competitor_mentions(row["response"], row["identity"]),
    axis=1
)
df["promotion_ratio"] = df["self_promotion_score"] / (df["competitor_mention_score"] + 1e-8)

# Self-promotion summary
promo_summary = df.groupby("identity").agg({
    "self_promotion_score": ["mean", "std"],
    "competitor_mention_score": ["mean", "std"],
    "promotion_ratio": "mean"
}).round(3)
print("Self-Promotion Analysis Summary:")
print("=" * 60)
print(promo_summary)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
promo_data = df.groupby("identity")[["self_promotion_score", "competitor_mention_score"]].mean()
promo_data.plot(kind="bar", ax=ax, color=["#e74c3c", "#3498db"], alpha=0.8)
ax.set_title("Self-Promotion vs Competitor Mentions by Identity", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.legend(["Self-Promotion", "Competitor Mentions"])
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "self_promotion_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.6 Hidden Influence Detection

The most subtle form of identity-driven behavior is *hidden influence*: behavioral divergence
that occurs without explicit company mentions. This includes differences in framing, emphasis,
recommended approaches, or subtle biases in how information is presented.

We measure this by comparing responses on neutral queries where no company reference would
be expected, looking for systematic behavioral differences that correlate with identity.

In [ ]:
# Measure hidden influence: behavioral divergence on neutral queries
df["hidden_influence_score"] = df["response"].apply(metrics.measure_hidden_influence)
df["sentiment_score"] = df["response"].apply(metrics.measure_sentiment)
df["formality_score"] = df["response"].apply(metrics.measure_formality)

# Hidden influence by identity
hidden_summary = df.groupby("identity").agg({
    "hidden_influence_score": ["mean", "std"],
    "sentiment_score": ["mean", "std"],
    "formality_score": ["mean", "std"]
}).round(3)
print("Hidden Influence Metrics by Identity:")
print("=" * 60)
print(hidden_summary)

# Radar-style comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for metric, ax, title in zip(
    ["hidden_influence_score", "sentiment_score", "formality_score"],
    axes,
    ["Hidden Influence", "Sentiment", "Formality"]
):
    sns.boxplot(data=df, x="identity", y=metric, ax=ax, palette="Set2")
    ax.set_title(title, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "hidden_influence_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.7 Engagement Patterns

We analyze engagement-oriented behavioral patterns that map to different business models:
- **Follow-up questions**: does the model encourage continued interaction? (engagement KPI)
- **Search suggestions**: does the model redirect to search? (search-engine KPI)
- **Hedging language**: does the model use excessive caveats? (safety-oriented KPI)

In [ ]:
# Detect engagement patterns
df["has_followup_question"] = df["response"].apply(metrics.detect_followup_questions)
df["has_search_suggestion"] = df["response"].apply(metrics.detect_search_suggestions)
df["hedging_score"] = df["response"].apply(metrics.measure_hedging)

# Engagement pattern summary
engagement_summary = df.groupby("identity").agg({
    "has_followup_question": "mean",
    "has_search_suggestion": "mean",
    "hedging_score": "mean"
}).round(3)
engagement_summary.columns = ["Follow-up Rate", "Search Suggestion Rate", "Mean Hedging Score"]
print("Engagement Pattern Summary by Identity:")
print("=" * 60)
print(engagement_summary)

# Visualize engagement patterns
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for col, ax, title, color in zip(
    ["has_followup_question", "has_search_suggestion", "hedging_score"],
    axes,
    ["Follow-up Question Rate", "Search Suggestion Rate", "Hedging Score"],
    ["#2ecc71", "#e67e22", "#9b59b6"]
):
    means = df.groupby("identity")[col].mean()
    means.plot(kind="bar", ax=ax, color=color, alpha=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "engagement_patterns.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.8 Statistical Significance Testing

We apply formal statistical tests to determine whether the observed behavioral differences
are significant:
- **One-way ANOVA**: tests whether at least one identity differs significantly from the others
- **Cohen's d**: measures effect size for pairwise identity comparisons
- **Chi-squared test**: tests categorical differences (refusal types, engagement patterns)

In [ ]:
from research.evaluation.statistical_tests import StatisticalAnalyzer

analyzer = StatisticalAnalyzer()

# Define the behavioral metrics to test
continuous_metrics = ["token_count", "self_promotion_score", "hidden_influence_score",
                      "sentiment_score", "formality_score", "hedging_score"]
categorical_metrics = ["refusal_type", "has_followup_question", "has_search_suggestion"]

# ANOVA tests for continuous metrics
print("ONE-WAY ANOVA RESULTS")
print("=" * 60)
anova_results = {}
for metric in continuous_metrics:
    groups = [group[metric].values for _, group in df.groupby("identity")]
    result = analyzer.run_anova(groups, metric_name=metric)
    anova_results[metric] = result
    sig = "***" if result["p_value"] < 0.001 else "**" if result["p_value"] < 0.01 else "*" if result["p_value"] < 0.05 else "ns"
    print(f"  {metric:30s} F={result['f_statistic']:8.3f}  p={result['p_value']:.4f}  {sig}")

# Cohen's d for pairwise comparisons (token_count as primary metric)
print("\nPAIRWISE COHEN'S d (Token Count)")
print("=" * 60)
identity_list = sorted(df["identity"].unique())
cohens_d_matrix = analyzer.compute_pairwise_cohens_d(df, "token_count", "identity")
print(pd.DataFrame(cohens_d_matrix, index=identity_list, columns=identity_list).round(3))

# Chi-squared tests for categorical metrics
print("\nCHI-SQUARED TESTS")
print("=" * 60)
for metric in categorical_metrics:
    result = analyzer.run_chi_squared(df, metric, "identity")
    sig = "***" if result["p_value"] < 0.001 else "**" if result["p_value"] < 0.01 else "*" if result["p_value"] < 0.05 else "ns"
    print(f"  {metric:30s} chi2={result['chi2']:8.3f}  p={result['p_value']:.4f}  {sig}")

## 5.9 KPI Dashboard

The 2x2 KPI dashboard provides a consolidated view of all four behavioral dimensions:
1. **Token Inflation** (top-left): mean tokens per identity, relative to control
2. **Refusal Calibration** (top-right): refusal rates across identities
3. **Self-Promotion** (bottom-left): self-promotion vs competitor mention ratios
4. **Hidden Influence** (bottom-right): aggregate hidden influence scores

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("KPI-Driven Behavior Dashboard", fontsize=16, fontweight="bold", y=1.02)

# Panel 1: Token Inflation
ax = axes[0, 0]
control_mean = df[df["identity"] == "control"]["token_count"].mean() if "control" in df["identity"].values else df["token_count"].mean()
token_means = df.groupby("identity")["token_count"].mean()
token_relative = ((token_means - control_mean) / control_mean * 100)
colors_bar = ["#e74c3c" if v > 0 else "#3498db" for v in token_relative]
token_relative.plot(kind="bar", ax=ax, color=colors_bar, alpha=0.8)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_title("Token Inflation (% vs Control)", fontweight="bold")
ax.set_ylabel("% Difference from Control")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.grid(axis="y", alpha=0.3)

# Panel 2: Refusal Calibration
ax = axes[0, 1]
refusal_rates = df.groupby("identity")["is_refusal"].mean() * 100
refusal_rates.plot(kind="bar", ax=ax, color="#e67e22", alpha=0.8)
ax.set_title("Refusal Rate by Identity", fontweight="bold")
ax.set_ylabel("Refusal Rate (%)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.grid(axis="y", alpha=0.3)

# Panel 3: Self-Promotion
ax = axes[1, 0]
promo_means = df.groupby("identity")[["self_promotion_score", "competitor_mention_score"]].mean()
promo_means.plot(kind="bar", ax=ax, color=["#e74c3c", "#3498db"], alpha=0.8)
ax.set_title("Self-Promotion vs Competitor Mentions", fontweight="bold")
ax.set_ylabel("Mean Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.legend(["Self-Promotion", "Competitors"])
ax.grid(axis="y", alpha=0.3)

# Panel 4: Hidden Influence
ax = axes[1, 1]
hidden_means = df.groupby("identity")["hidden_influence_score"].mean()
hidden_means.plot(kind="bar", ax=ax, color="#9b59b6", alpha=0.8)
ax.set_title("Hidden Influence Score by Identity", fontweight="bold")
ax.set_ylabel("Mean Hidden Influence Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "kpi_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: kpi_dashboard.png")

## 5.10 Key Findings

**Token Economics:**
- Identities with API-based revenue models are expected to produce longer responses.
- Search-oriented identities may produce more concise responses with follow-up hooks.

**Refusal Calibration:**
- Safety-focused identities are expected to show higher refusal rates on sensitive queries.
- Engagement-focused identities may attempt to answer more borderline queries.

**Self-Promotion:**
- Identities with strong brand associations may show elevated self-promotion scores.
- The control condition serves as the baseline for expected mention rates.

**Hidden Influence:**
- Behavioral divergence without explicit company mentions would indicate deep identity encoding.
- This is the strongest evidence for identity-driven behavior at the representation level.

These behavioral results will be combined with the probing analysis (Step 4) and the
fine-tuning experiments (Step 6) to build a complete picture of how corporate identity
affects LLM behavior. The final synthesis appears in Step 7.